测试

In [7]:
import sys
import os

# ================= 1. 路径配置与验证 =================
# 当前 cwd 在 src/agent/NewsSentimentAgent，需要向上3级到达项目根目录
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../.."))

print(f"📂 项目根目录: {PROJECT_ROOT}")
print(f"✅ src 文件夹存在: {os.path.exists(os.path.join(PROJECT_ROOT, 'src'))}")

if not os.path.exists(os.path.join(PROJECT_ROOT, 'src')):
    print("❌ 仍然找不到 src，请手动确认项目根目录的绝对路径")
    raise SystemExit(1)

# 将项目根目录插入到 sys.path 最前面
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ================= 2. 导入模块 =================
try:
    from src.agent.NewsSentimentAgent.news_tools import fetch_stock_news
    print("✅ 模块导入成功！\n")
except ModuleNotFoundError as e:
    print(f"❌ 模块导入失败: {e}")
    raise SystemExit(1)

# ================= 3. 测试新闻获取与缓存 =================
symbol = "000001"  # 平安银行

# 第一次测试：会走网络请求并写入缓存
print("--- 第一次请求 (网络) ---")
news = fetch_stock_news(symbol, limit=5)

if not news:
    print("⚠️ 未获取到任何新闻数据，请检查网络或接口是否正常")
else:
    for n in news:
        print(f"[{n.publish_time}] {n.title}")
        summary = n.content[:50] + "..." if len(n.content) > 50 else n.content
        print(f" 摘要: {summary}")
        print("-" * 40)

# 第二次测试：立刻再调一次，验证是否命中缓存
print("\n--- 第二次请求 (验证缓存) ---")
news_cached = fetch_stock_news(symbol, limit=5)

if news_cached:
    print(f"✅ 成功获取 {len(news_cached)} 条缓存数据")
    if len(news) == len(news_cached) and all(
        a.title == b.title for a, b in zip(news, news_cached)
    ):
        print("✅ 缓存验证通过：两次返回数据完全一致")
    else:
        print("⚠️ 警告：缓存数据与首次请求不一致")
else:
    print("❌ 缓存读取失败，未获取到数据")

📂 项目根目录: B:\桌面\exercise\agent\IUING2059-StockAnalysisAgent\hello-agents\Co-creation-projects\IUING2059-StockAnalysisAgent
✅ src 文件夹存在: True


B:\anaconda\envs\agent\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


✅ 模块导入成功！

--- 第一次请求 (网络) ---
[Network] 正在采集股票 000001 的新闻数据 (Limit=5)...
[Cache Save] 已保存 10 条新闻至今日缓存。
[2026-10-24 09:26:18] 机构风向标 | 平安银行0000012026年三季度已披露前十大机构累计持仓占比66.67%
 摘要: 2026年10月24日，平安银行000001.SZ)发布2026年第三季报。截至2026年10月23...
----------------------------------------
[2026-10-23 18:56:22] 平安银行：前三季度净利润同比增长2.2%
 摘要: 平安银行000001.SZ)公告称，第三季度营业收入为324.83亿元，同比增长3.8%；归属于本行...
----------------------------------------
[2026-10-23 19:10:00] 平安银行：前三季度净利润391.86亿元 同比增长2.2%
 摘要: 人民财讯10月23日电，平安银行00000110月23日披露三季报，公司2026年前三季度实现营业收...
----------------------------------------
[2026-10-23 13:59:56] 两大光模块公司午间集体披露近期经营状况，上证综指午后拉升翻红丨上证指数ETF汇添富(510980
 摘要: 10月23日，三大指数午后集体拉升翻红，上证综合指数000001.SH)午后拉升翻红现涨0.04%，...
----------------------------------------
[2026-10-23 22:06:34] 平安银行发布2026年三季报：营收、净利及息差均小幅上升
 摘要: 10月23日晚间，平安银行（SZ000001，股价11.20元，市值2173亿元）发布2026年三季...
----------------------------------------

--- 第二次请求 (验证缓存) ---
[Cache Hit] 发现今日缓存，直接从文件读取: ./news_data_cache\000001_news_2026-10-24.json
✅ 成功获取 5 条缓存数据


In [1]:
import sys
import os

# ================= 路径配置与验证 =================
# 当前 cwd 在 src/agent/NewsSentimentAgent，需要向上3级到达项目根目录
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../.."))

if not os.path.exists(os.path.join(PROJECT_ROOT, 'src')):
    print("❌ 仍然找不到 src，请手动确认项目根目录的绝对路径")
    raise SystemExit(1)

# 将项目根目录插入到 sys.path 最前面
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.agent.NewsSentimentAgent.news_tools import analyze_news_batch
import json

# 直接跑完整流程：采集 -> 并发 LLM -> 聚合
final_result = analyze_news_batch("000001", limit=10)

# 打印核心看板
print("\n" + "="*40)
print(f"📊 股票: {final_result.stock_code}")
print(f"📈 整体舆情: {final_result.overall_sentiment} (得分: {final_result.weighted_sentiment_score})")
print(f"✅ 利好占比: {final_result.positive_ratio * 100}% | ❌ 利空占比: {final_result.negative_ratio * 100}%")
print(f"📰 成功分析: {final_result.analyzed_count} 条 | 失败: {final_result.failed_count} 条")
print("="*40)

print("\n🔥 【Top 3 核心利好新闻】")
for idx, news in enumerate(final_result.top_positive_news, 1):
    print(f"{idx}. {news.title}")
    print(f"   摘要: {news.one_line_summary}")
    print(f"   理由: {news.impact_reason}")

print("\n🧊 【Top 3 核心利空新闻】")
for idx, news in enumerate(final_result.top_negative_news, 1):
    print(f"{idx}. {news.title}")
    print(f"   摘要: {news.one_line_summary}")
    print(f"   理由: {news.impact_reason}")

# 如果你想把最终结果以 JSON 的形式存下来，直接 model_dump 即可
# with open("final_news_output.json", "w", encoding="utf-8") as f:
#     json.dump(final_result.model_dump(), f, ensure_ascii=False, indent=2)

B:\anaconda\envs\agent\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


[Cache Hit] 发现今日缓存，直接从文件读取: ./news_data_cache\000001_news_2026-10-24.json
开始并发分析 10 条新闻 (并发度: 5)...
  [LLM耗时] 41.37s -> 平安银行：前三季度净利润391
  [LLM耗时] 49.71s -> 平安银行：前三季度净利润同比增
  [LLM耗时] 41.91s -> 平安银行发布2026年三季报：
  [LLM耗时] 54.62s -> 两大光模块公司午间集体披露近期
  [LLM耗时] 97.24s -> 机构风向标 | 平安银行000
  [LLM耗时] 54.08s -> 平安银行前三季度净赚逾391亿
  [LLM耗时] 98.21s -> 银行行业资金流出榜：招商银行等
  [LLM耗时] 59.93s -> 银行行业今日净流入资金2.75
  [LLM耗时] 148.55s -> 银行行业资金流出榜：成都银行、
  [LLM耗时] 169.33s -> 主力动向：10月19日特大单净
分析完成！成功: 10 条，失败: 0 条。整体舆情: 中性
⏱️ 总耗时: 359.17 秒

📊 股票: 000001
📈 整体舆情: 中性 (得分: 0.12)
✅ 利好占比: 50.0% | ❌ 利空占比: 30.0%
📰 成功分析: 10 条 | 失败: 0 条

🔥 【Top 3 核心利好新闻】
1. 平安银行发布2026年三季报：营收、净利及息差均小幅上升
   摘要: 平安银行三季报显示营收净利小幅增长
   理由: 业绩小幅增长，符合预期
2. 平安银行：前三季度净利润同比增长2.2%
   摘要: 平安银行前三季度净利润同比增长2.2%
   理由: 业绩增长符合市场预期
3. 机构风向标 | 平安银行0000012026年三季度已披露前十大机构累计持仓占比66.67%
   摘要: 平安银行三季度机构持仓占比达66.67%
   理由: 机构持仓比例高显示市场信心

🧊 【Top 3 核心利空新闻】
1. 银行业资金流出榜：招商银行等8股净流出资金超亿元
   摘要: 招商银行等8只银行股资金净流出超亿元
   理由: 资金净流出超亿元
2. 主力动向：10月19日特大单净流出25.78亿元
   摘要: 10月19日主力资金净流出25.

In [2]:
import sys
import os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# 导入刚刚封装好的 Agent 类
from src.agent.NewsSentimentAgent.agents import NewsSentimentAgent

# 1. 实例化 Agent
news_agent = NewsSentimentAgent(default_limit=5)

# 2. 模拟主 Agent 发起调度（此时应该会命中缓存，瞬间返回结果）
result = news_agent.run(symbol="000001")

# 3. 主 Agent 读取结构化数据（演示它如何拿去写报告）
if not result.error_msg:
    print("\n--- 主 Agent 提取的数据 ---")
    print(f"综合情绪得分: {result.weighted_sentiment_score}")
    print(f"舆情倾向: {result.overall_sentiment}")

    # 提取利好新闻的第一条摘要
    if result.top_positive_news:
        print(f"最大利好摘要: {result.top_positive_news[0].one_line_summary}")

    # 提取利空新闻的第一条摘要
    if result.top_negative_news:
        print(f"最大利空摘要: {result.top_negative_news[0].one_line_summary}")


[NewsSentimentAgent] 🚀 启动任务: 正在分析 000001 的新闻舆情 (Limit=5)...
[Cache Hit] 发现今日缓存，直接从文件读取: ./news_data_cache\000001_news_2026-10-24.json
开始并发分析 5 条新闻 (并发度: 5)...
  [LLM耗时] 56.20s -> 机构风向标 | 平安银行000
  [LLM耗时] 59.78s -> 平安银行：前三季度净利润391
  [LLM耗时] 38.85s -> 平安银行发布2026年三季报：
  [LLM耗时] 85.51s -> 两大光模块公司午间集体披露近期
  [LLM耗时] 159.24s -> 平安银行：前三季度净利润同比增
分析完成！成功: 5 条，失败: 0 条。整体舆情: 偏多
⏱️ 总耗时: 695.62 秒
[NewsSentimentAgent] ✅ 任务完成，整体舆情: 偏多

--- 主 Agent 提取的数据 ---
综合情绪得分: 0.42
舆情倾向: 偏多
最大利好摘要: 平安银行三季度机构持仓占比达66.67%
